## Notebook 1: Isolating Disjoint Trees

The underlying idea is simple. Just want two small-ish model family trees that are disjoint.

In [ ]:
import datasets

ds = datasets.load_dataset("modelbiome/ai_ecosystem_withmodelcards")

/Users/vedantpathak/Developer/projects/packages/looker/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
ds["train"].filter(lambda e: e["model_id"].lower().startswith("google"))

Filter: 100%|██████████| 1860411/1860411 [00:28<00:00, 65476.70 examples/s]


Dataset({
    features: ['Unnamed: 0', 'model_id', 'likes', 'trendingScore', 'private', 'downloads', 'tags', 'pipeline_tag', 'library_name', 'createdAt', 'regions', 'licenses', 'arxiv_papers', 'datasets', 'parent_model', 'finetune_parent', 'quantized_parent', 'adapter_parent', 'merge_parent', 'languages', 'modelCard', 'ratelimit_retries', 'exception_raised'],
    num_rows: 1070
})

In [17]:
googl = _

In [18]:
googl[0]

{'Unnamed: 0': 12,
 'model_id': 'google/gemma-3n-E4B-it',
 'likes': 545,
 'trendingScore': 91.0,
 'private': False,
 'downloads': 260307,
 'tags': "['transformers', 'safetensors', 'gemma3n', 'image-text-to-text', 'automatic-speech-recognition', 'automatic-speech-translation', 'audio-text-to-text', 'video-text-to-text', 'conversational', 'arxiv:1905.07830', 'arxiv:1905.10044', 'arxiv:1911.11641', 'arxiv:1904.09728', 'arxiv:1705.03551', 'arxiv:1911.01547', 'arxiv:1907.10641', 'arxiv:1903.00161', 'arxiv:2210.03057', 'arxiv:2502.12404', 'arxiv:2411.19799', 'arxiv:2009.03300', 'arxiv:2502.21228', 'arxiv:2311.12022', 'arxiv:2403.07974', 'arxiv:2108.07732', 'arxiv:2107.03374', 'base_model:google/gemma-3n-E4B', 'base_model:finetune:google/gemma-3n-E4B', 'license:gemma', 'endpoints_compatible', 'region:us']",
 'pipeline_tag': 'image-text-to-text',
 'library_name': 'transformers',
 'createdAt': '2025-06-03T18:17:07.000Z',
 'regions': "['us']",
 'licenses': "['gemma']",
 'arxiv_papers': "['1905.0

In [ ]:
ds["train"].filter(lambda e: e["model_id"] == "google/gemma-3n-E4B")[0]

Filter: 100%|██████████| 1860411/1860411 [00:28<00:00, 66095.45 examples/s]


Dataset({
    features: ['Unnamed: 0', 'model_id', 'likes', 'trendingScore', 'private', 'downloads', 'tags', 'pipeline_tag', 'library_name', 'createdAt', 'regions', 'licenses', 'arxiv_papers', 'datasets', 'parent_model', 'finetune_parent', 'quantized_parent', 'adapter_parent', 'merge_parent', 'languages', 'modelCard', 'ratelimit_retries', 'exception_raised'],
    num_rows: 1
})

In [28]:
import dataclasses

@dataclasses.dataclass
class Tree:
    value: str
    children: list[Tree] = dataclasses.field(default_factory=list)

def assemble_tree(
    ds: datasets.Dataset,
    root: str,
    max_depth: int = 3,
    max_width: int = 2,
) -> Tree:
    node = Tree(value=root)
    
    if max_depth == 1:
        return node
    
    for i, child in enumerate(ds.filter(lambda row: root in row["parent_model"])):
        if i >= max_width:
            break
        
        child_node = assemble_tree(
            ds, child["model_id"], max_depth=max_depth - 1, max_width=max_width
        )
        
        node.children.append(child_node)

    return node

In [32]:
gemma_tree = assemble_tree(ds["train"], "google/gemma-3n-E4B")

Ok, now that we've assembled a family tree for Gemma, let's try to find one for Llama.

In [30]:
ds["train"].filter(lambda e: e["model_id"].lower().startswith("meta-llama/llama"))[0]

Filter: 100%|██████████| 1860411/1860411 [00:29<00:00, 62223.39 examples/s]


{'Unnamed: 0': 22,
 'model_id': 'meta-llama/Llama-3.1-8B-Instruct',
 'likes': 4284,
 'trendingScore': 54.0,
 'private': False,
 'downloads': 5522768,
 'tags': "['transformers', 'safetensors', 'llama', 'text-generation', 'facebook', 'meta', 'pytorch', 'llama-3', 'conversational', 'en', 'de', 'fr', 'it', 'pt', 'hi', 'es', 'th', 'arxiv:2204.05149', 'base_model:meta-llama/Llama-3.1-8B', 'base_model:finetune:meta-llama/Llama-3.1-8B', 'license:llama3.1', 'autotrain_compatible', 'text-generation-inference', 'endpoints_compatible', 'region:us']",
 'pipeline_tag': 'text-generation',
 'library_name': 'transformers',
 'createdAt': '2024-07-18T08:56:00.000Z',
 'regions': "['us']",
 'licenses': "['llama3.1']",
 'arxiv_papers': "['2204.05149']",
 'datasets': '[]',
 'parent_model': "['meta-llama/Llama-3.1-8B']",
 'finetune_parent': "['meta-llama/Llama-3.1-8B']",
 'quantized_parent': '[]',
 'adapter_parent': '[]',
 'merge_parent': '[]',
 'languages': "['English', 'German', 'French', 'Italian', 'Portug

meta-llama/Llama-3.1-8B should be the root node. Checking.

In [31]:
ds["train"].filter(lambda e: e["model_id"] == "meta-llama/Llama-3.1-8B")[0]

Filter: 100%|██████████| 1860411/1860411 [00:30<00:00, 61493.93 examples/s]


{'Unnamed: 0': 121,
 'model_id': 'meta-llama/Llama-3.1-8B',
 'likes': 1690,
 'trendingScore': 14.0,
 'private': False,
 'downloads': 743001,
 'tags': "['transformers', 'safetensors', 'llama', 'text-generation', 'facebook', 'meta', 'pytorch', 'llama-3', 'en', 'de', 'fr', 'it', 'pt', 'hi', 'es', 'th', 'arxiv:2204.05149', 'license:llama3.1', 'autotrain_compatible', 'text-generation-inference', 'endpoints_compatible', 'region:us']",
 'pipeline_tag': 'text-generation',
 'library_name': 'transformers',
 'createdAt': '2024-07-14T22:20:15.000Z',
 'regions': "['us']",
 'licenses': "['llama3.1']",
 'arxiv_papers': "['2204.05149']",
 'datasets': '[]',
 'parent_model': '[]',
 'finetune_parent': '[]',
 'quantized_parent': '[]',
 'adapter_parent': '[]',
 'merge_parent': '[]',
 'languages': "['English', 'German', 'French', 'Italian', 'Portuguese', 'Hindi', 'Spanish', 'Thai']",
 'modelCard': '---\nlanguage:\n- en\n- de\n- fr\n- it\n- pt\n- hi\n- es\n- th\npipeline_tag: text-generation\ntags:\n- facebo

I was right. Let's collect its tree.

In [34]:
llama_tree = assemble_tree(ds["train"], "meta-llama/Llama-3.1-8B")

Filter: 100%|██████████| 1860411/1860411 [00:29<00:00, 63821.22 examples/s]


Now it's time to dump the trees to a JSON file.

In [37]:
import json
import pathlib


def trees_to_json(roots: list[Tree], filename: str) -> None:
    json_dict: dict[str, list[str]] = {}
    
    def _add_to_list(node: Tree, running_list: list[str]) -> None:
        for child in node.children:
            running_list.append(child.value)
            _add_to_list(child, running_list)
            
    for root in roots:
        json_dict[root.value] = [root.value]
        _add_to_list(root, json_dict[root.value])
    
    fnm = pathlib.Path(filename).with_suffix(".json")
    
    with open(fnm, "w") as f:
        json.dump(json_dict, f, indent=2)

trees_to_json([gemma_tree, llama_tree], "trees.json")
        